# Task 3 — run 1 gradient-norm and peak-memory diagnostic

Run 1 (`t3_shreya_unet_128_20261002_023102`) did not log gradient norms or peak GPU memory. This notebook
measures them **on the submitted model** by resuming its epoch-80 checkpoint, discriminators and optimizer
states included, and training **one more epoch** (7,038 steps) with run 1's exact losses and data pipeline
at epoch 80's learning rate (1.905e-5).

- Logged every step: generator gradient L2 norm (G_AB + G_BA, before the optimizer step, never clipped; the
  same `grad_G` definition as the later runs), both discriminators' gradient norms, and the losses.
- Peak GPU memory for a training step (batch size 1, fp32), which depends only on the model and batch size.
- **The updated weights are discarded.** Nothing on Drive is overwritten; the submission is unchanged.
- The replay buffers start empty (run 1's were not saved), so the first 50 discriminator steps differ.

Writes to `reproducibility/raw_logs/shreya_akotiya/task3_gan/`:
`t3_shreya_unet_128_20261002_023102_graddiag.csv` (per step) and `..._graddiag.json` (summary).

Use an **A100 GPU runtime** (run 1's hardware), then *Runtime → Run all*. About 10–15 minutes.


In [1]:
import os, sys
from pathlib import Path


def lab1_root() -> Path:
    """Repo root, in priority order: LAB1_ROOT env var, the Drive copy on Colab,
    then a local clone found by walking up. Same convention as Tasks 1-2, so a
    run is relocatable instead of tied to one Drive account."""
    if os.environ.get("LAB1_ROOT"):
        return Path(os.environ["LAB1_ROOT"]).expanduser().resolve()
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/DATA266_Lab1")
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "task3_gan").is_dir() and (cand / "README.md").exists():
            return cand
    raise FileNotFoundError(
        "repo root not found - set LAB1_ROOT, e.g. "
        "os.environ['LAB1_ROOT'] = '/content/drive/MyDrive/DATA266_Lab1'")


ROOT = lab1_root()
os.environ["LAB1_ROOT"] = str(ROOT)       # child processes inherit it
print("LAB1_ROOT:", ROOT)

Mounted at /content/drive
LAB1_ROOT: /content/drive/MyDrive/DATA266_Lab1


In [2]:
import json, math, os, random, shutil, time, itertools
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = Path(os.environ["LAB1_ROOT"])   # resolved in the cell above
RUN_ID = "t3_shreya_unet_128_20261002_023102"
CKPT = ROOT / "task3_gan/shreya_akotiya/checkpoints" / f"{RUN_ID}_epoch080.pt"
OUT = ROOT / "reproducibility/raw_logs/shreya_akotiya/task3_gan"
SEED, STEPS, LOG_EVERY = 42, 7038, 500
# run 1's settings (config.yaml + notebook override cell 8)
IMG, LOAD, LR, BETAS = 256, 286, 2e-4, (0.5, 0.999)
LAMBDA_CYCLE, LAMBDA_IDENTITY, POOL = 10.0, 0.5, 50
LR_EPOCH80 = LR * (1.0 - (79 - 60) / 21.0)   # LambdaLR value used during epoch 80 = 1.905e-5

assert torch.cuda.is_available(), "use a GPU runtime"
DEVICE = "cuda"
GPU = torch.cuda.get_device_name(0)
print(GPU, torch.__version__)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


Tesla T4 2.11.0+cu130


In [3]:
# Copy the images to local disk once (reading 7,038 files from Drive every step is slow)
# On Colab, stage the images on local disk (reading 7,038 files from Drive
# every step is slow). Off Colab the repo copy is already local.
LOCAL = Path("/content/t3_data") if "google.colab" in sys.modules else ROOT / "task3_gan/data"
for d in ("photo_jpg", "monet_jpg"):
    if not (LOCAL / d).exists():
        shutil.copytree(ROOT / "task3_gan/data" / d, LOCAL / d)
print({d: len(list((LOCAL / d).glob("*.jpg"))) for d in ("photo_jpg", "monet_jpg")})


{'photo_jpg': 7038, 'monet_jpg': 300}


In [4]:
# ---- run 1 generator, copied from task3_cyclegan.ipynb (cells 17-18) ---------------------------

class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
                  nn.InstanceNorm2d(out_ch), nn.ReLU(inplace=True)]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip):
        return torch.cat([self.model(x), skip], dim=1)


class UNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, use_dropout=True):
        super().__init__()
        self.down1 = UNetDown(in_ch, ngf, normalize=False)
        self.down2 = UNetDown(ngf, ngf * 2)
        self.down3 = UNetDown(ngf * 2, ngf * 4)
        self.down4 = UNetDown(ngf * 4, ngf * 8)
        self.down5 = UNetDown(ngf * 8, ngf * 8)
        self.down6 = UNetDown(ngf * 8, ngf * 8)
        self.down7 = UNetDown(ngf * 8, ngf * 8, normalize=False)
        dropout = 0.5 if use_dropout else 0.0
        self.up1 = UNetUp(ngf * 8, ngf * 8, dropout=dropout)
        self.up2 = UNetUp(ngf * 16, ngf * 8, dropout=dropout)
        self.up3 = UNetUp(ngf * 16, ngf * 8, dropout=dropout)
        self.up4 = UNetUp(ngf * 16, ngf * 4)
        self.up5 = UNetUp(ngf * 8, ngf * 2)
        self.up6 = UNetUp(ngf * 4, ngf)
        self.final = nn.Sequential(nn.ConvTranspose2d(ngf * 2, out_ch, 4, stride=2, padding=1), nn.Tanh())

    def forward(self, x):
        d1 = self.down1(x); d2 = self.down2(d1); d3 = self.down3(d2); d4 = self.down4(d3)
        d5 = self.down5(d4); d6 = self.down6(d5); d7 = self.down7(d6)
        u1 = self.up1(d7, d6); u2 = self.up2(u1, d5); u3 = self.up3(u2, d4)
        u4 = self.up4(u3, d3); u5 = self.up5(u4, d2); u6 = self.up6(u5, d1)
        return self.final(u6)


In [5]:
# ---- run 1's discriminator, dataset, replay buffer (copied from task3_cyclegan.ipynb) ----------

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=3, ndf=64, n_layers=3):
        super().__init__()
        layers = [nn.Conv2d(in_ch, ndf, 4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True)]
        mult = 1
        for i in range(1, n_layers):
            prev, mult = mult, min(2 ** i, 8)
            layers += [nn.Conv2d(ndf * prev, ndf * mult, 4, stride=2, padding=1, bias=False),
                       nn.InstanceNorm2d(ndf * mult), nn.LeakyReLU(0.2, inplace=True)]
        layers += [nn.Conv2d(ndf * mult, 1, 4, stride=1, padding=1)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class ImageDataset(Dataset):
    def __init__(self, root_a, root_b, transform):
        self.files_a = sorted(Path(root_a).glob("*.jpg"))
        self.files_b = sorted(Path(root_b).glob("*.jpg"))
        self.transform = transform

    def __len__(self):
        return max(len(self.files_a), len(self.files_b))

    def __getitem__(self, idx):
        a = Image.open(self.files_a[idx % len(self.files_a)]).convert("RGB")
        b = Image.open(self.files_b[random.randint(0, len(self.files_b) - 1)]).convert("RGB")
        return {"A": self.transform(a), "B": self.transform(b)}


class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size, self.data = max_size, []

    def push_and_pop(self, data):
        result = []
        for element in data:
            element = element.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(element); result.append(element)
            elif random.uniform(0, 1) > 0.5:
                idx = random.randint(0, self.max_size - 1)
                result.append(self.data[idx].clone()); self.data[idx] = element
            else:
                result.append(element)
        return torch.cat(result, dim=0)


train_tf = transforms.Compose([
    transforms.Resize(LOAD, transforms.InterpolationMode.BICUBIC), transforms.RandomCrop(IMG),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])
loader = DataLoader(ImageDataset(LOCAL / "photo_jpg", LOCAL / "monet_jpg", train_tf),
                    batch_size=1, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
print(len(loader), "batches per epoch")


7038 batches per epoch


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [6]:
# ---- load run 1's epoch-80 state ---------------------------------------------------------------
ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
assert ck["epoch"] == 80
G_AB, G_BA = UNetGenerator().to(DEVICE), UNetGenerator().to(DEVICE)   # photo->Monet, Monet->photo
D_A, D_B = PatchGANDiscriminator().to(DEVICE), PatchGANDiscriminator().to(DEVICE)   # photo, Monet
for name, m in (("G_AB", G_AB), ("G_BA", G_BA), ("D_A", D_A), ("D_B", D_B)):
    m.load_state_dict(ck[name])
n_params = sum(p.numel() for m in (G_AB, G_BA, D_A, D_B) for p in m.parameters())
assert n_params == 84_967_560, n_params

opt_G = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=LR, betas=BETAS)
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)
for name, opt in (("opt_G", opt_G), ("opt_D_A", opt_D_A), ("opt_D_B", opt_D_B)):
    opt.load_state_dict(ck[name])
    for g in opt.param_groups:
        g["lr"] = LR_EPOCH80
del ck
print(f"loaded epoch 80, {n_params:,} params, lr {LR_EPOCH80:.4g}")

mse, l1 = nn.MSELoss(), nn.L1Loss()
pool_A, pool_B = ReplayBuffer(POOL), ReplayBuffer(POOL)

def grad_norm(params):
    return torch.nn.utils.clip_grad_norm_(params, float("inf")).item()   # measures only, never clips


loaded epoch 80, 84,967,560 params, lr 1.905e-05


In [7]:
# ---- one epoch with run 1's exact training step, instrumented ------------------------------------
G_params = list(G_AB.parameters()) + list(G_BA.parameters())
for m in (G_AB, G_BA, D_A, D_B):
    m.train()
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
rows, t0 = [], time.time()
for step, batch in enumerate(loader, 1):
    real_A, real_B = batch["A"].to(DEVICE), batch["B"].to(DEVICE)
    valid = torch.ones_like(D_A(real_A)); fake = torch.zeros_like(D_A(real_A))   # as in run 1 (unused)

    opt_G.zero_grad()
    loss_id_B = l1(G_AB(real_B), real_B) * LAMBDA_CYCLE * LAMBDA_IDENTITY
    loss_id_A = l1(G_BA(real_A), real_A) * LAMBDA_CYCLE * LAMBDA_IDENTITY
    fake_B = G_AB(real_A); p = D_B(fake_B); loss_GAN_AB = mse(p, torch.ones_like(p))
    fake_A = G_BA(real_B); p = D_A(fake_A); loss_GAN_BA = mse(p, torch.ones_like(p))
    loss_cyc_A = l1(G_BA(fake_B), real_A) * LAMBDA_CYCLE
    loss_cyc_B = l1(G_AB(fake_A), real_B) * LAMBDA_CYCLE
    loss_G = loss_GAN_AB + loss_GAN_BA + loss_cyc_A + loss_cyc_B + loss_id_A + loss_id_B
    loss_G.backward()
    gn_G = grad_norm(G_params)
    opt_G.step()

    opt_D_A.zero_grad()
    p_real, p_fake = D_A(real_A), D_A(pool_A.push_and_pop(fake_A.detach()))
    loss_D_A = 0.5 * (mse(p_real, torch.ones_like(p_real)) + mse(p_fake, torch.zeros_like(p_fake)))
    loss_D_A.backward(); gn_D_A = grad_norm(D_A.parameters()); opt_D_A.step()

    opt_D_B.zero_grad()
    p_real, p_fake = D_B(real_B), D_B(pool_B.push_and_pop(fake_B.detach()))
    loss_D_B = 0.5 * (mse(p_real, torch.ones_like(p_real)) + mse(p_fake, torch.zeros_like(p_fake)))
    loss_D_B.backward(); gn_D_B = grad_norm(D_B.parameters()); opt_D_B.step()

    rows.append({"step": step, "grad_norm_G": gn_G, "grad_norm_D_A": gn_D_A, "grad_norm_D_B": gn_D_B,
                 "loss_G": loss_G.item(), "loss_D_A": loss_D_A.item(), "loss_D_B": loss_D_B.item(),
                 "loss_cycle": loss_cyc_A.item() + loss_cyc_B.item()})
    if step % LOG_EVERY == 0:
        print(f"step {step}/{STEPS} | grad_G {gn_G:.2f} | G {loss_G.item():.3f} | D_A {loss_D_A.item():.4f} | "
              f"D_B {loss_D_B.item():.4f} | {time.time() - t0:.0f}s")
    if step == STEPS:
        break
torch.cuda.synchronize()
peak_alloc = torch.cuda.max_memory_allocated() / 1e9
peak_reserved = torch.cuda.max_memory_reserved() / 1e9
elapsed = time.time() - t0
print(f"done: {len(rows)} steps in {elapsed:.0f}s, peak allocated {peak_alloc:.3f} GB, reserved {peak_reserved:.3f} GB")


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


step 500/7038 | grad_G 38.50 | G 2.423 | D_A 0.0419 | D_B 0.0468 | 102s
step 1000/7038 | grad_G 34.62 | G 2.392 | D_A 0.1427 | D_B 0.0528 | 207s
step 1500/7038 | grad_G 36.04 | G 1.964 | D_A 0.2219 | D_B 0.0557 | 313s
step 2000/7038 | grad_G 36.35 | G 2.661 | D_A 0.1353 | D_B 0.0731 | 419s
step 2500/7038 | grad_G 34.78 | G 2.337 | D_A 0.1203 | D_B 0.0468 | 525s
step 3000/7038 | grad_G 40.84 | G 2.274 | D_A 0.0518 | D_B 0.1003 | 631s
step 3500/7038 | grad_G 27.36 | G 1.861 | D_A 0.1505 | D_B 0.0969 | 737s
step 4000/7038 | grad_G 38.92 | G 2.156 | D_A 0.1214 | D_B 0.0631 | 842s
step 4500/7038 | grad_G 51.84 | G 2.464 | D_A 0.0723 | D_B 0.1185 | 948s
step 5000/7038 | grad_G 36.91 | G 2.340 | D_A 0.0949 | D_B 0.1276 | 1054s
step 5500/7038 | grad_G 30.81 | G 2.358 | D_A 0.0706 | D_B 0.0565 | 1160s
step 6000/7038 | grad_G 57.74 | G 2.422 | D_A 0.0775 | D_B 0.0465 | 1266s
step 6500/7038 | grad_G 41.69 | G 1.910 | D_A 0.0857 | D_B 0.0628 | 1372s
step 7000/7038 | grad_G 34.19 | G 2.873 | D_A 0.

In [8]:
# ---- save per-step values and summary (the trained weights are NOT saved) -----------------------
df = pd.DataFrame(rows)
df.to_csv(OUT / f"{RUN_ID}_graddiag.csv", index=False)
g = df.grad_norm_G
roll = g.rolling(100, min_periods=20)
spikes = int((g > roll.mean().shift(1) + 3 * roll.std().shift(1)).sum())   # > mean + 3 std of previous 100 steps
summary = {
    "run_id": RUN_ID, "checkpoint": CKPT.name, "what": "1 diagnostic epoch from the epoch-80 checkpoint; weights discarded",
    "steps": len(df), "lr": LR_EPOCH80, "batch_size": 1, "precision": "fp32", "gpu": GPU, "torch": torch.__version__,
    "grad_norm_G": {"mean": g.mean(), "median": g.median(), "p95": g.quantile(0.95), "max": g.max(),
                    "max_step": int(df.step[g.idxmax()]), "spikes_3sd": spikes},
    "grad_norm_D_A_mean": df.grad_norm_D_A.mean(), "grad_norm_D_B_mean": df.grad_norm_D_B.mean(),
    "non_finite_steps": int((~np.isfinite(df.drop(columns="step").values)).any(axis=1).sum()),
    "peak_memory_allocated_gb": peak_alloc, "peak_memory_reserved_gb": peak_reserved,
    "images_per_sec": len(df) / elapsed,
    "loss_means": {k: df[k].mean() for k in ("loss_G", "loss_D_A", "loss_D_B", "loss_cycle")},
}
(OUT / f"{RUN_ID}_graddiag.json").write_text(json.dumps(summary, indent=2, default=float))
print(json.dumps(summary, indent=2, default=float))


{
  "run_id": "t3_shreya_unet_128_20261002_023102",
  "checkpoint": "t3_shreya_unet_128_20261002_023102_epoch080.pt",
  "what": "1 diagnostic epoch from the epoch-80 checkpoint; weights discarded",
  "steps": 7038,
  "lr": 1.9047619047619046e-05,
  "batch_size": 1,
  "precision": "fp32",
  "gpu": "Tesla T4",
  "torch": "2.11.0+cu130",
  "grad_norm_G": {
    "mean": 38.55937375721521,
    "median": 36.072471618652344,
    "p95": 58.745368766784665,
    "max": 399.4892883300781,
    "max_step": 5039,
    "spikes_3sd": 127
  },
  "grad_norm_D_A_mean": 7.917352125112155,
  "grad_norm_D_B_mean": 4.985887515473752,
  "non_finite_steps": 0,
  "peak_memory_allocated_gb": 1.775904256,
  "peak_memory_reserved_gb": 1.895825408,
  "images_per_sec": 4.737647201752093,
  "loss_means": {
    "loss_G": 2.4636202963382963,
    "loss_D_A": 0.09396993015257896,
    "loss_D_B": 0.07139668509609728,
    "loss_cycle": 0.8013079075709634
  }
}
